## bronzeテーブルのパイプラインをDatabricks Auto Loaderにて作成

In [0]:
%run ../../変数設定

### CSVファイルをそのまま取り込むための、空テーブルを作成

In [0]:
# 用意したいカラム
# TODO：過去実装からバッククォートで囲む必要がある特殊なカラム名を探す
schema = """
    record_id STRING,
    usage_start_time STRING,
    usage_end_time STRING,
    usage_quantity STRING,
    sku STRING,
    workspace_id STRING,
    identity_metadata STRING
"""

# Bronzeテーブルを作成
# TODO: USING deltaは書かなくてもDeltaテーブルになること
# TODO：Deltaテーブルとは
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {bronze_usage_table_path} (
        {schema},
        _datasource STRING,
        _ingest_timestamp timestamp
    )
    """
)

# # テーブル作り直し
# spark.sql(
#     f"""
#     REPLACE TABLE {bronze_usage_table_path} (
#         {schema},
#         _datasource STRING,
#         _ingest_timestamp timestamp
#     )
#     """
# )



In [0]:
from pyspark.sql.functions import col, from_utc_timestamp

# ソースからデータを読み込む
df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "False")
    .option("quote", '"')
    .option("escape", '"')  # ダブルクォートをエスケープ
    .option("multiLine", "true")  # 複数行にまたがるフィールドがある場合必要
    .load(usage_csv_path)
)

# 監査列として`_datasource`列と`_ingest_timestamp`列を追加
df = (
    df.select("*", "_metadata")
    .withColumn("_datasource", df["_metadata.file_path"])
    .withColumn(
        "_ingest_timestamp",
        from_utc_timestamp(col("_metadata.file_modification_time"), "Asia/Tokyo"),
    )
    .drop("_metadata")
)

In [0]:
df.display()

TODO：VSCodeで文字化けしたときの対処法を記載
## カラムのズレを修正

原因：値の中にカンマが入っている  
対処法：
```python
    .option("quote", '"')
    .option("escape", '"')  # ダブルクォートをエスケープ
```

### bronzeテーブルにdata frameを書き込み

In [0]:
(
    df.write.format("delta")
        .mode("append")
        .saveAsTable(bronze_usage_table_path)
)


In [0]:
# データが書き込まれたことを確認
display(spark.table(bronze_usage_table_path))

列追加を検知・追加列を表示したいとき

In [0]:
# 既存テーブルがあるか確認
table_exists = spark.catalog.tableExists(bronze_usage_table_path)

if table_exists:
    # 既存テーブルのスキーマ取得
    existing_df = spark.table(bronze_usage_table_path)
    existing_columns = set(existing_df.columns)

    # 今回書き込み予定のスキーマ
    incoming_columns = set(df.columns)

    # 追加された列を検出（既存にない列）
    new_columns = incoming_columns - existing_columns

    if new_columns:
        raise Exception(
            f"新しい列が検出されました: {new_columns}\n"
            "意図しないスキーマ変更の可能性があります。\n"
            "内容を確認の上、mergeSchemaを明示的に有効化してください。"
        )

    # 問題なければ通常書き込み
    (
        df.write.format("delta")
          .mode("append")
          .saveAsTable(bronze_usage_table_path)
    )

else:
    # 初回作成時
    (
        df.write.format("delta")
          .mode("overwrite")
          .saveAsTable(bronze_usage_table_path)
    )
